In [15]:
import pandas as pd
from linearmodels.panel import PanelOLS
import numpy as np

REPEATING BASELINE REGRESSION BUT WITH MACRO VARIABLES

We begin by performing our baseline regression and inspecting our results.

In [16]:
baseline_reg = pd.read_csv("csv_data/BASELINE1.csv")
baseline_reg.columns

Index(['costat', 'curcd', 'datafmt', 'indfmt', 'consol', 'sic', 'datadate',
       'gvkey', 'conm', 'tic', 'fyear', 'at', 'ceq', 'dltt', 'lse', 'ni',
       'revt', 'xrd', 'csho', 'prcc_f', 'sich', 'mkt_cap', 'industry',
       'boardid', 'companyid', 'datestartrole', 'directorid', 'directorname',
       'companyname', 'rolename', 'dateendrole', 'datestartroleflag',
       'dateendroleflag', 'seniority', 'UG', 'top20_ug', 'MBA', 'top20_mba',
       'PhD', 'top20_phd', 'MD', 'top20_md', 'Master's', 'top20_masters',
       'dob', 'gender', 'diversitynetworklabel', 're', 'volatility', 'roa',
       'adjusted_roa', 'firm_size', 'equity_capital', 'charter_value',
       'retained_earnings', 'rd_intensity', 'ceo_age', 'ceo_gender',
       'masters_factor', 'md_factor', 'phd_factor', 'mba_factor', 'ug_factor'],
      dtype='object')

In [17]:
# world bank data and us burea of economic analysis (BEA)
gdp_growth = {
    2015: 2.9, 2016: 1.8, 2017: 2.5, 2018: 3.0,
    2019: 2.6, 2020: -2.2, 2021: 6.1, 2022: 2.5,
    2023: 2.9, 2024: 2.8, 2025: 2.1
}

baseline_reg['gdp_growth'] = baseline_reg['fyear'].map(gdp_growth)

In [18]:
print(baseline_reg['gender'].value_counts(dropna=False))
print(baseline_reg['gender'].unique())

gender
M      958
F       56
NaN     18
Name: count, dtype: int64
['F' 'M' nan]


In [19]:
baseline_reg['CEO_gender'] = baseline_reg['gender'].map({'M': 0, 'F': 1})
print(baseline_reg['CEO_gender'].value_counts(dropna=False))

CEO_gender
0.0    958
1.0     56
NaN     18
Name: count, dtype: int64


In [20]:
reg_vars = ['adjusted_roa', 'roa', 'ug_factor', 'mba_factor', 'phd_factor', 
            'md_factor', 'masters_factor', 'firm_size', 'equity_capital', 
            'charter_value', 'retained_earnings', 'rd_intensity', 
            'volatility', 'ceo_age', 'CEO_gender', 'gdp_growth']

# Drop NaNs on regression variables only
baseline_reg_clean = baseline_reg.dropna(subset=reg_vars)
print(baseline_reg_clean.shape)
print(baseline_reg_clean['CEO_gender'].value_counts(dropna=False))

(943, 65)
CEO_gender
0.0    891
1.0     52
Name: count, dtype: int64


In [21]:
print(baseline_reg_clean['gender'].value_counts(dropna=False))

gender
M    891
F     52
Name: count, dtype: int64


In [28]:
# remove overage CEOs (> 100 years old, not included in sample)
baseline_reg_clean = baseline_reg_clean[baseline_reg_clean['ceo_age'] < 100]
print(baseline_reg_clean.shape)

(931, 65)


In [29]:
panel_data = baseline_reg_clean.set_index(['gvkey', 'fyear'])

In [30]:
baseline_reg_clean.shape

(931, 65)

TBU

In [31]:
# Simplest possible model first
model = PanelOLS.from_formula(
    '''adjusted_roa ~ mba_factor + phd_factor + md_factor + 
       masters_factor + ug_factor +
       firm_size + equity_capital + charter_value + retained_earnings + rd_intensity +
       volatility + gdp_growth + ceo_age + CEO_gender +
       EntityEffects + TimeEffects''',
    data=panel_data,
    drop_absorbed=True,
    check_rank=False
)

result = model.fit(cov_type='clustered', cluster_entity=True)
print(result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:           adjusted_roa   R-squared:                        0.2728
Estimator:                   PanelOLS   R-squared (Between):             -1.0218
No. Observations:                 931   R-squared (Within):              -4.9708
Date:                Fri, May 08 2026   R-squared (Overall):             -3.1299
Time:                        18:18:01   Log-likelihood                    1296.4
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      21.903
Entities:                         149   P-value                           0.0000
Avg Obs:                       6.2483   Distribution:                  F(13,759)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             9.0980
                            

In [32]:
biotech = baseline_reg_clean[baseline_reg_clean['industry'] == 'Biotech']
semis = baseline_reg_clean[baseline_reg_clean['industry'] == 'Semiconductors/Hardware']
software = baseline_reg_clean[baseline_reg_clean['industry'] == 'Software']

In [33]:
panel_bio = biotech.set_index(['gvkey', 'fyear'])

model = PanelOLS.from_formula(
    '''adjusted_roa ~ mba_factor + phd_factor + md_factor + 
       masters_factor + ug_factor +
       firm_size + equity_capital + charter_value + retained_earnings + rd_intensity +
       volatility + gdp_growth + ceo_age + CEO_gender +
       EntityEffects + TimeEffects''',
    data=panel_bio,
    drop_absorbed=True,
    check_rank=False
)

result = model.fit(cov_type='clustered', cluster_entity=True)
print(result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:           adjusted_roa   R-squared:                        0.4593
Estimator:                   PanelOLS   R-squared (Between):             -2.0021
No. Observations:                 330   R-squared (Within):              -11.758
Date:                Fri, May 08 2026   R-squared (Overall):             -7.1528
Time:                        18:19:24   Log-likelihood                    481.93
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      16.400
Entities:                          56   P-value                           0.0000
Avg Obs:                       5.8929   Distribution:                  F(13,251)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             21.489
                            

In [34]:
biotech.value_counts("UG")

UG
1.0    237
0.0     93
Name: count, dtype: int64

In [35]:
panel_semis = semis.set_index(['gvkey', 'fyear'])

model = PanelOLS.from_formula(
    '''adjusted_roa ~ mba_factor + phd_factor + md_factor + 
       masters_factor + ug_factor +
       firm_size + equity_capital + charter_value + retained_earnings + rd_intensity +
       volatility + gdp_growth + ceo_age + CEO_gender +
       EntityEffects + TimeEffects''',
    data=panel_semis,
    drop_absorbed=True,
    check_rank=False
)

result = model.fit(cov_type='clustered', cluster_entity=True)
print(result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:           adjusted_roa   R-squared:                        0.2336
Estimator:                   PanelOLS   R-squared (Between):             -0.2139
No. Observations:                 388   R-squared (Within):              -4.1489
Date:                Fri, May 08 2026   R-squared (Overall):             -1.9203
Time:                        18:21:01   Log-likelihood                    596.03
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      7.2677
Entities:                          55   P-value                           0.0000
Avg Obs:                       7.0545   Distribution:                  F(13,310)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             14.905
                            

In [131]:
panel_software = software.set_index(['gvkey', 'fyear'])

model = PanelOLS.from_formula(
    '''adjusted_roa ~ mba_factor + phd_factor + md_factor + 
       masters_factor + ug_factor +
       firm_size + equity_capital + charter_value + retained_earnings + rd_intensity +
       volatility + gdp_growth + ceo_age + CEO_gender +
       EntityEffects + TimeEffects''',
    data=panel_software,
    drop_absorbed=True,
    check_rank=False
)

result = model.fit(cov_type='clustered', cluster_entity=True)
print(result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:           adjusted_roa   R-squared:                        0.3877
Estimator:                   PanelOLS   R-squared (Between):             -1.7549
No. Observations:                 214   R-squared (Within):              -14.707
Date:                Sat, Apr 25 2026   R-squared (Overall):             -8.2737
Time:                        12:22:04   Log-likelihood                    307.52
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      7.4039
Entities:                          39   P-value                           0.0000
Avg Obs:                       5.4872   Distribution:                  F(13,152)
Min Obs:                       1.0000                                           
Max Obs:                       11.000   F-statistic (robust):             6.6082
                            

In [132]:
desc_stats = baseline_reg_clean[['adjusted_roa', 'ug_factor', 'mba_factor', 
                                   'phd_factor', 'md_factor', 'masters_factor',
                                   'firm_size', 'equity_capital', 'charter_value',
                                   'retained_earnings', 'rd_intensity', 'volatility',
                                   'ceo_age', 'CEO_gender', 'gdp_growth',
                                   'UG', 'top20_ug', 'MBA', 'top20_mba',
                                   'PhD', 'top20_phd', 'MD', 'top20_md',
                                   "Master's", 'top20_masters']].describe().T

print(desc_stats)

                   count       mean       std        min        25%  \
adjusted_roa       943.0   0.005120  0.103328  -0.585825  -0.044997   
ug_factor          943.0   0.014332  0.743538  -1.766358  -0.510675   
mba_factor         943.0   0.015337  0.915715  -0.653556  -0.618238   
phd_factor         943.0   0.011576  0.988875  -0.491438  -0.330529   
md_factor          943.0   0.014905  0.991015  -0.511685  -0.275306   
masters_factor     943.0   0.033572  0.973399  -1.184203  -0.677365   
firm_size          943.0   9.279721  1.341655   6.561793   8.208657   
equity_capital     943.0   0.465432  0.203866   0.001293   0.328264   
charter_value      943.0   1.571578  1.005119  -0.579732   0.901636   
retained_earnings  943.0   0.132061  0.488557  -2.804056  -0.076440   
rd_intensity       943.0   0.092652  0.083551   0.000000   0.036141   
volatility         943.0   0.024488  0.009854   0.004184   0.017662   
ceo_age            943.0  58.507953  9.873923  35.000000  53.000000   
CEO_ge

In [137]:
print(baseline_reg_clean['fyear'].nunique())

11


In [10]:
desc_stats_baseline = baseline_reg_clean[['adjusted_roa', 
                                           'ug_factor', 'mba_factor', 'phd_factor', 
                                           'md_factor', 'masters_factor',
                                           'firm_size', 'equity_capital', 'charter_value',
                                           'retained_earnings', 'rd_intensity', 'volatility',
                                           'ceo_age', 'CEO_gender', 'gdp_growth']].describe().T

print(desc_stats_baseline)

                   count       mean       std        min        25%  \
adjusted_roa       943.0   0.005120  0.103328  -0.585825  -0.044997   
ug_factor          943.0   0.014332  0.743538  -1.766358  -0.510675   
mba_factor         943.0   0.015337  0.915715  -0.653556  -0.618238   
phd_factor         943.0   0.011576  0.988875  -0.491438  -0.330529   
md_factor          943.0   0.014905  0.991015  -0.511685  -0.275306   
masters_factor     943.0   0.033572  0.973399  -1.184203  -0.677365   
firm_size          943.0   9.279721  1.341655   6.561793   8.208657   
equity_capital     943.0   0.465432  0.203866   0.001293   0.328264   
charter_value      943.0   1.571578  1.005119  -0.579732   0.901636   
retained_earnings  943.0   0.132061  0.488557  -2.804056  -0.076440   
rd_intensity       943.0   0.092652  0.083551   0.000000   0.036141   
volatility         943.0   0.024488  0.009854   0.004184   0.017662   
ceo_age            943.0  58.507953  9.873923  35.000000  53.000000   
CEO_ge

In [11]:
print(baseline_reg_clean['industry'].value_counts())
print(baseline_reg_clean.groupby('industry')['gvkey'].nunique())

industry
Semiconductors/Hardware    389
Biotech                    340
Software                   214
Name: count, dtype: int64
industry
Biotech                    57
Semiconductors/Hardware    55
Software                   39
Name: gvkey, dtype: int64


In [12]:
print(baseline_reg_clean[baseline_reg_clean['ceo_age'] > 100][['directorname', 'dob', 'fyear', 'ceo_age', 'tic']])

           directorname         dob  fyear  ceo_age   tic
35          Barry Allan  1900-01-01   2025    125.0   FIG
134         Paddy Nicol  1900-01-01   2025    125.0   OGN
342    Rafael Sotomayor  1900-01-01   2025    125.0  NXPI
742      Rob Humphryson  1900-01-01   2017    117.0   NVO
743      Rob Humphryson  1900-01-01   2018    118.0   NVO
744      Rob Humphryson  1900-01-01   2019    119.0   NVO
745      Rob Humphryson  1900-01-01   2020    120.0   NVO
747  Mike Spreadborough  1900-01-01   2022    122.0   NVO
748  Mike Spreadborough  1900-01-01   2023    123.0   NVO
749  Mike Spreadborough  1900-01-01   2024    124.0   NVO
750  Mike Spreadborough  1900-01-01   2025    125.0   NVO
847         Jared Kelly  1900-01-01   2025    125.0   ONC
